In [ ]:
# Example 1: Basic ASN.1 Data Types and Encoding
from pyasn1.type import univ, char, useful, tag
from pyasn1.codec.der import encoder, decoder
from pyasn1.codec.native import decoder as native_decoder

# Create basic ASN.1 data types
integer_val = univ.Integer(12345)
boolean_val = univ.Boolean(True)
octet_string = univ.OctetString(b'Hello ASN.1')
utf8_string = char.UTF8String('Testing pyasn1')

print("=== Basic ASN.1 Data Types ===")
print(f"Integer: {integer_val} (type: {type(integer_val)})")
print(f"Boolean: {boolean_val}")
print(f"Octet String: {octet_string}")
print(f"UTF8 String: {utf8_string}")

# Encode to DER format
encoded_int = encoder.encode(integer_val)
encoded_bool = encoder.encode(boolean_val)
encoded_octet = encoder.encode(octet_string)

print(f"\nDER Encoded Integer: {encoded_int.hex()}")
print(f"DER Encoded Boolean: {encoded_bool.hex()}")
print(f"DER Encoded Octet String: {encoded_octet.hex()}")

# Decode back from DER
decoded_int, _ = decoder.decode(encoded_int, asn1Spec=univ.Integer())
print(f"Decoded Integer: {decoded_int}")
print("Example 1 completed successfully!")

In [ ]:
# Example 2: Complex ASN.1 Structures - Sequence and Choice
from pyasn1.type import univ, namedtype, namedval, constraint
from pyasn1.codec.der import encoder, decoder

# Define a Person structure using SEQUENCE
class Person(univ.Sequence):
    componentType = namedtype.NamedTypes(
        namedtype.NamedType('name', char.UTF8String()),
        namedtype.NamedType('age', univ.Integer()),
        namedtype.OptionalNamedType('email', char.UTF8String())
    )

# Define a Contact Choice
class ContactMethod(univ.Choice):
    componentType = namedtype.NamedTypes(
        namedtype.NamedType('phone', char.UTF8String()),
        namedtype.NamedType('email', char.UTF8String()),
        namedtype.NamedType('address', char.UTF8String())
    )

print("=== Complex ASN.1 Structures ===")

# Create and populate structures
person = Person()
person.setComponentByName('name', 'Alice Johnson')
person.setComponentByName('age', 30)
person.setComponentByName('email', 'alice@example.com')

contact = ContactMethod()
contact.setComponentByName('email', 'contact@company.com')

print(f"Person: {person}")
print(f"Contact Method: {contact}")
print(f"Contact choice: {contact.getName()} = {contact.getComponent()}")

# Encode and decode the structures  
encoded_person = encoder.encode(person)
encoded_contact = encoder.encode(contact)

decoded_person, _ = decoder.decode(encoded_person, asn1Spec=Person())
print(f"Decoded Person - Name: {decoded_person.getComponentByName('name')}")
print("Example 2 completed successfully!")

In [ ]:
# Example 3: ASN.1 Collections - SEQUENCE OF and SET OF
from pyasn1.type import univ, constraint

print("=== ASN.1 Collections ===")

# SEQUENCE OF - ordered collection (like arrays)
class IntegerList(univ.SequenceOf):
    componentType = univ.Integer()

# SET OF - unordered collection (like sets)  
class StringSet(univ.SetOf):
    componentType = char.UTF8String()

# Create and populate collections
int_list = IntegerList()
int_list.setComponentByPosition(0, 10)
int_list.setComponentByPosition(1, 20) 
int_list.setComponentByPosition(2, 30)
int_list.setComponentByPosition(3, 40)

string_set = StringSet()
string_set.setComponentByPosition(0, 'apple')
string_set.setComponentByPosition(1, 'banana')
string_set.setComponentByPosition(2, 'cherry')

print(f"Integer List: {int_list}")
print(f"String Set: {string_set}")
print(f"List length: {len(int_list)}")
print(f"Set length: {len(string_set)}")

# Iterate through collections
print("Integer List contents:")
for i in range(len(int_list)):
    print(f"  [{i}]: {int_list.getComponentByPosition(i)}")

print("String Set contents:")  
for i in range(len(string_set)):
    print(f"  [{i}]: {string_set.getComponentByPosition(i)}")

# Encode collections
encoded_list = encoder.encode(int_list)
encoded_set = encoder.encode(string_set)
print(f"Encoded list size: {len(encoded_list)} bytes")
print("Example 3 completed successfully!")

In [ ]:
# Example 4: ASN.1 Tags and Constraints
from pyasn1.type import tag, constraint

print("=== ASN.1 Tags and Constraints ===")

# Create integers with different tagging
implicit_tagged = univ.Integer().subtype(
    implicitTag=tag.Tag(tag.tagClassContext, tag.tagFormatSimple, 1)
)

explicit_tagged = univ.Integer().subtype(
    explicitTag=tag.Tag(tag.tagClassContext, tag.tagFormatConstructed, 2)
)

# Apply value constraints - use subtypeSpec keyword argument
constrained_int = univ.Integer().subtype(
    subtypeSpec=constraint.ValueRangeConstraint(1, 100)
)

# String with size constraints
constrained_string = char.UTF8String().subtype(
    subtypeSpec=constraint.ValueSizeConstraint(1, 50)
)

print("Setting values...")
implicit_tagged = implicit_tagged.clone(42)
explicit_tagged = explicit_tagged.clone(123)
constrained_int = constrained_int.clone(75)
constrained_string = constrained_string.clone("Valid string")

print(f"Implicit tagged (context 1): {implicit_tagged}")
print(f"Explicit tagged (context 2): {explicit_tagged}")  
print(f"Constrained integer (1-100): {constrained_int}")
print(f"Constrained string (max 50 chars): {constrained_string}")

# Test constraint validation
from pyasn1.type.error import ValueConstraintError
try:
    invalid_int = univ.Integer().subtype(subtypeSpec=constraint.ValueRangeConstraint(1, 10))
    invalid_int = invalid_int.clone(15)  # This should raise an error
except ValueConstraintError as e:
    assert "ValueRangeConstraint" in str(e), f"unexpected ValueConstraintError message: {e}"
    print(f"Constraint validation caught: {type(e).__name__}")
else:
    raise AssertionError(f"15 was accepted by a 1..10 ValueRangeConstraint: {invalid_int!r}")

# Encode tagged values
encoded_implicit = encoder.encode(implicit_tagged)
encoded_explicit = encoder.encode(explicit_tagged)
print(f"Implicit tag encoding: {encoded_implicit.hex()}")
print(f"Explicit tag encoding: {encoded_explicit.hex()}")
print("Example 4 completed successfully!")

## Example 4: ASN.1 Tags and Constraints

Demonstrating ASN.1 tagging mechanisms and value constraints for data validation.

In [ ]:
# Example 5: Real-world ASN.1 Application - Simplified Certificate Structure
from pyasn1.type import useful
from pyasn1.codec.der import encoder, decoder
import base64

print("=== Real-world ASN.1 - Simplified Certificate Structure ===")

# Define a simplified certificate structure (avoiding nested complexity)
class SimplifiedCertificate(univ.Sequence):
    componentType = namedtype.NamedTypes(
        namedtype.NamedType('version', univ.Integer()),
        namedtype.NamedType('serialNumber', univ.Integer()),
        namedtype.NamedType('signatureAlgorithm', univ.ObjectIdentifier()),
        namedtype.NamedType('issuer', char.UTF8String()),
        namedtype.NamedType('notBefore', useful.GeneralizedTime()),
        namedtype.NamedType('notAfter', useful.GeneralizedTime()),
        namedtype.NamedType('subject', char.UTF8String()),
        namedtype.NamedType('publicKeyAlgorithm', univ.ObjectIdentifier()),
        namedtype.NamedType('publicKeyData', univ.BitString())
    )

# Create a sample certificate structure
cert = SimplifiedCertificate()
cert.setComponentByName('version', 3)  # v3
cert.setComponentByName('serialNumber', 123456789)
cert.setComponentByName('signatureAlgorithm', univ.ObjectIdentifier('1.2.840.113549.1.1.11'))  # SHA256withRSA
cert.setComponentByName('issuer', 'CN=Test CA,O=Example Org,C=US')
cert.setComponentByName('notBefore', useful.GeneralizedTime('20240101120000Z'))
cert.setComponentByName('notAfter', useful.GeneralizedTime('20251231235959Z'))
cert.setComponentByName('subject', 'CN=example.com,O=Example Inc,C=US')
cert.setComponentByName('publicKeyAlgorithm', univ.ObjectIdentifier('1.2.840.113549.1.1.1'))  # RSA
cert.setComponentByName('publicKeyData', univ.BitString("'0011010101101100'B"))  # Sample public key bits

print(f"Certificate version: {cert.getComponentByName('version')}")
print(f"Serial number: {cert.getComponentByName('serialNumber')}")
print(f"Signature algorithm: {cert.getComponentByName('signatureAlgorithm')}")
print(f"Issuer: {cert.getComponentByName('issuer')}")
print(f"Not before: {cert.getComponentByName('notBefore')}")
print(f"Not after: {cert.getComponentByName('notAfter')}")
print(f"Subject: {cert.getComponentByName('subject')}")
print(f"Public key algorithm: {cert.getComponentByName('publicKeyAlgorithm')}")

# Encode the certificate
encoded_cert = encoder.encode(cert)
print(f"\nEncoded certificate size: {len(encoded_cert)} bytes")
print(f"Encoded hex (first 50 bytes): {encoded_cert[:50].hex()}")

# Base64 encode for transport
b64_cert = base64.b64encode(encoded_cert)
print(f"Base64 encoded (first 80 chars): {b64_cert[:80].decode()}...")

# Decode it back
decoded_cert, remainder = decoder.decode(encoded_cert, asn1Spec=SimplifiedCertificate())
print(f"\nDecoded certificate subject: {decoded_cert.getComponentByName('subject')}")
print(f"Decoded serial number: {decoded_cert.getComponentByName('serialNumber')}")

# Demonstrate practical use: creating multiple certificates
print(f"\n=== Batch Certificate Creation ===")
certificates = []
for i in range(3):
    batch_cert = SimplifiedCertificate()
    batch_cert.setComponentByName('version', 3)
    batch_cert.setComponentByName('serialNumber', 1000 + i)
    batch_cert.setComponentByName('signatureAlgorithm', univ.ObjectIdentifier('1.2.840.113549.1.1.11'))
    batch_cert.setComponentByName('issuer', 'CN=Batch CA')
    batch_cert.setComponentByName('notBefore', useful.GeneralizedTime('20240101000000Z'))
    batch_cert.setComponentByName('notAfter', useful.GeneralizedTime('20250101000000Z'))
    batch_cert.setComponentByName('subject', f'CN=device-{i}.example.com')
    batch_cert.setComponentByName('publicKeyAlgorithm', univ.ObjectIdentifier('1.2.840.113549.1.1.1'))
    batch_cert.setComponentByName('publicKeyData', univ.BitString(f"'{i:08b}'B"))
    certificates.append(batch_cert)

print(f"Created {len(certificates)} certificates")
for i, cert in enumerate(certificates):
    encoded = encoder.encode(cert)
    print(f"  Certificate {i}: subject={cert.getComponentByName('subject')}, size={len(encoded)} bytes")

print("Example 5 completed successfully!")

In [ ]:
# Additional: Testing pyasn1 compatibility and edge cases
from pyasn1.codec.ber import encoder as ber_encoder, decoder as ber_decoder

print("=== ASN.1 Compatibility Testing ===")

# Test different encoding rules (BER vs DER)
test_data = univ.Integer(987654321)

der_encoded = encoder.encode(test_data)
ber_encoded = ber_encoder.encode(test_data)

print(f"DER encoded: {der_encoded.hex()}")
print(f"BER encoded: {ber_encoded.hex()}")
print(f"Encodings match: {der_encoded == ber_encoded}")

# Test NULL type
null_val = univ.Null()
encoded_null = encoder.encode(null_val)
print(f"NULL encoded: {encoded_null.hex()}")

# Test ENUMERATED type  
class Status(univ.Enumerated):
    namedValues = namedval.NamedValues(
        ('active', 1),
        ('inactive', 2),
        ('pending', 3)
    )

status = Status('active')
print(f"Status: {status} (value: {int(status)})")

# Test BIT STRING with named bits
flags = univ.BitString("'10110000'B")
print(f"Bit string: {flags}")
print(f"Bit string as binary: {flags.asBinary()}")

# Performance test with larger structures
large_sequence = univ.SequenceOf(componentType=univ.Integer())
for i in range(100):
    large_sequence.setComponentByPosition(i, i * i)

encoded_large = encoder.encode(large_sequence)
print(f"Large sequence (100 integers) encoded size: {len(encoded_large)} bytes")
print("Compatibility testing completed successfully!")